In [ ]:
# CELL 1: Setup
import sys
sys.path.append('..')

import os
import json
import torch
import numpy as np
import random
import matplotlib.pyplot as plt

from configs.config import Config
from data.splits import get_datasets
from models.maml_segmentation import MAMLSegmentation, MAMLTrainer

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

Config.create_dirs()
print(f"✓ Device: {Config.DEVICE}")

Verifying BraTS2020 dataset via KaggleHub...
✓ Device: mps
✓ Data path: /Users/yuzheli/.cache/kagglehub/datasets/awsaf49/brats20-dataset-training-validation/versions/1/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData


In [ ]:
# CELL 2: Data Loading
train_dataset, val_dataset, test_dataset = get_datasets(Config)

print(f"✓ Train samples: {len(train_dataset)}")
print(f"✓ Val samples:   {len(val_dataset)}")

✓ Filename already correct


In [ ]:
# CELL 3: Create MAML Model + Load Pretrained Weights
model = MAMLSegmentation(
    encoder_name=Config.ENCODER_NAME,
    in_channels=Config.IN_CHANNELS,
    num_classes=Config.NUM_CLASSES,
).to(Config.DEVICE)

# Load pretrained baseline weights
checkpoint = torch.load(
    os.path.join(Config.CHECKPOINT_DIR, 'best_model.pth'),
    weights_only=False,
    map_location=Config.DEVICE,
)
model.model.load_state_dict(checkpoint['model_state_dict'], strict=False)

print(f"✓ Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print("✓ Loaded pretrained weights from baseline")

✓ Train: 258, Val: 74, Test: 37


In [ ]:
# CELL 4: MAML Training
trainer = MAMLTrainer(
    model=model,
    config=Config,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
)

history = trainer.train(
    num_tasks=1000,
    k_shot=5,
    n_query=Config.N_QUERY,
)

✓ Valid patients: 258/258
✓ Valid patients: 258/258
✓ Valid patients: 74/74
✓ Valid patients: 37/37


In [ ]:
# CELL 5: Training Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['meta_train_loss'], alpha=0.3, color='blue')
# Smoothed version (rolling average)
window = 50
if len(history['meta_train_loss']) >= window:
    smoothed = np.convolve(history['meta_train_loss'],
                           np.ones(window)/window, mode='valid')
    axes[0].plot(smoothed, color='blue', label='Meta Train Loss (smoothed)')
axes[0].set_xlabel('Task')
axes[0].set_ylabel('Loss')
axes[0].set_title('MAML Meta-Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['meta_val_loss'], marker='o')
axes[1].set_xlabel('Checkpoint (×100 tasks)')
axes[1].set_ylabel('Loss')
axes[1].set_title('MAML Validation Loss')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(Config.RESULTS_DIR, 'maml_training_curve.png'), dpi=150)
plt.show()

In [ ]:
# CELL 6: K-Shot Evaluation
k_shot_results = trainer.evaluate_k_shot(
    k_values=Config.K_SHOT_VALUES,
    num_tasks=50,
)

print(f"\n{'='*50}")
print("MAML K-SHOT RESULTS")
print(f"{'='*50}")
for k, result in k_shot_results.items():
    print(f"  k={k:>2}: Dice = {result['mean']:.4f} ± {result['std']:.4f}")
print(f"{'='*50}")

# Save
results_path = os.path.join(Config.RESULTS_DIR, 'maml_kshot_results.json')
with open(results_path, 'w') as f:
    json.dump({str(k): {'mean': float(v['mean']), 'std': float(v['std'])}
               for k, v in k_shot_results.items()}, f, indent=4)
print(f"✓ Saved to: {results_path}")

✓ Train batches: 3225
✓ Val batches: 925
✓ Test batches: 463


### Analysis — Why MAML Underperforms

MAML Analysis: Why It Didn't Beat the Baseline

1. IMPLEMENTATION LIMITATION: Our MAML uses first-order approximation
   (FOMAML). The meta_train_step() trains on concatenated support+query
   data rather than backpropagating through the inner loop adaptation.
   This reduces MAML to standard training.

2. ARCHITECTURAL CONSTRAINT: True second-order MAML requires functional
   forward passes (swapping parameters at each layer). SMP's U-Net
   doesn't support this natively — would need learn2learn or higher.

3. THEORETICAL INSIGHT: MAML's value is learning a good initialization.
   But our model already starts from ImageNet-pretrained ResNet34, which
   is already a strong initialization. This limits the room for MAML
   to improve, consistent with recent literature on pretrained backbones
   reducing meta-learning's added value.

CONCLUSION: The negative result is itself informative — strong transfer
learning from pretrained encoders can match or exceed meta-learning
approaches in the few-shot medical imaging setting.